# Airbnb Europe — Data preparation (VS Code)

In [ ]:
from pathlib import Path
import pandas as pd
import duckdb
from IPython.display import display

# Detect the project root when VS Code starts in /notebooks or the repository root.
HERE = Path.cwd().resolve()
PROJECT_ROOT = next(
    (p for p in [HERE, *HERE.parents] if (p / 'data').is_dir()),
    HERE.parent if HERE.name.lower() == 'notebooks' else HERE
)
DATA_DIR = PROJECT_ROOT / 'data'
Listing_Data_Path = DATA_DIR / 'listings'
Review_Data_Path = DATA_DIR / 'reviews'
Output_Data_Path = DATA_DIR / 'processed'
EDA_DIR = PROJECT_ROOT / 'reports' / 'data_quality'
TEMP_DIR = PROJECT_ROOT / 'duckdb_temp'
for path in (Output_Data_Path, EDA_DIR, TEMP_DIR):
    path.mkdir(parents=True, exist_ok=True)

print('Project root:', PROJECT_ROOT)
print('Listings folder:', Listing_Data_Path)
print('Reviews folder:', Review_Data_Path)
print('Processed output:', Output_Data_Path)
if not Listing_Data_Path.exists() or not Review_Data_Path.exists():
    raise FileNotFoundError('Create data/listings and data/reviews and copy your original CSV files into them.')

Initialise DuckDB

In [ ]:
# Persistent local DuckDB database; keep review data on disk rather than in pandas.
con = duckdb.connect(str(PROJECT_ROOT / 'airbnb_reviews.duckdb'))
con.execute("SET memory_limit = '12GB'")
con.execute('SET threads = 4')
safe_temp = str(TEMP_DIR).replace("'", "''")
con.execute(f"SET temp_directory = '{safe_temp}'")
print('DuckDB connected successfully!')

In [ ]:
#Verify that DuckDB can read the files
reviews_files = sorted(
    Path(Review_Data_Path).glob("*_reviews.csv")
)

print(f"Found {len(reviews_files)} CSV files")

assert len(reviews_files) > 0, "No review files found!"

# Inspect the first file
first_file = str(reviews_files[0])

test_reviews = con.execute("""
    SELECT *
    FROM read_csv_auto(
        ?,
        all_varchar = true
    )
    LIMIT 5
""", [first_file]).df()

display(test_reviews)
print(test_reviews.columns.tolist())

Creating Dataframes for Listings and Reviews



1. Listings:

In [ ]:
#Dictionary of locations
location_country = {
    "Amsterdam": "Netherlands",
    "Antwerp": "Belgium",
    "Athens": "Greece",
    "Barcelona": "Spain",
    "Bergamo": "Italy",
    "Berlin": "Germany",
    "Bologna": "Italy",
    "Bordeaux": "France",
    "Bristol": "United Kingdom",
    "Brussels": "Belgium",
    "Copenhagen": "Denmark",
    "Crete": "Greece",
    "Dublin": "Ireland",
    "Edinburgh": "United Kingdom",
    "Euskadi": "Spain",
    "Florence": "Italy",
    "Geneva": "Switzerland",
    "Ghent": "Belgium",
    "Girona": "Spain",
    "Greater Manchester": "United Kingdom",
    "Lisbon": "Portugal",
    "London": "United Kingdom",
    "Lyon": "France",
    "Madrid": "Spain",
    "Mallorca": "Spain",
    "Menorca": "Spain",
    "Milan": "Italy",
    "Munich": "Germany",
    "Naples": "Italy",
    "Oslo": "Norway",
    "Paris": "France",
    "Porto": "Portugal",
    "Prague": "Czech Republic",
    "Puglia": "Italy",
    "Riga": "Latvia",
    "Rome": "Italy",
    "Rotterdam": "Netherlands",
    "Sevilla": "Spain",
    "Sicily": "Italy",
    "South Aegean": "Greece",
    "The Hague": "Netherlands",
    "Thessaloniki": "Greece",
    "Valencia": "Spain",
    "Vaud": "Switzerland",
    "Venice": "Italy",
    "Vienna": "Austria"
}


In [ ]:
#Function to combine CSV files to dataframes
def combine_airbnb_files(folder_path, file_type):
    folder = Path(folder_path)
    # Detect files ending with _listings.csv or _reviews.csv
    csv_files = sorted(folder.glob(f"*_{file_type}.csv"))
    print(f"\nFound {len(csv_files)} {file_type} files.")

    if not csv_files:
        raise FileNotFoundError(
            f"No matching CSV files found in {folder}"
        )

    dataframes = []
    file_summary = []

    for file in csv_files:
        # Extract location from filename
        # Example: Greater_Manchester_listings.csv
        # Result: Greater Manchester
        location = (
            file.stem
            .removesuffix(f"_{file_type}")
            .replace("_", " ")
        )
        country = location_country.get(location, "Unknown")

        # Read file
        df = pd.read_csv(
            file,
            low_memory=False
        )

        # Add geographic and source identifiers
        df["location"] = location
        df["country"] = country
        df["source_file"] = file.name

        dataframes.append(df)

        file_summary.append({
            "location": location,
            "country": country,
            "file": file.name,
            "rows": len(df),
            "columns": len(df.columns)
        })

        print(
            f"Loaded {file.name}: "
            f"{len(df):,} rows"
        )

    # Concatenate all files vertically
    master_df = pd.concat(
        dataframes,
        ignore_index=True,
        sort=False
    )

    summary_df = pd.DataFrame(file_summary)

    print(f"\n{file_type.upper()} MASTER CREATED")
    print(f"Total rows: {len(master_df):,}")
    print(f"Total columns: {master_df.shape[1]}")
    print(f"Locations: {master_df['location'].nunique()}")
    print(f"Countries: {master_df['country'].nunique()}")

    return master_df, summary_df

In [ ]:
#Listings master dataframe
listings_master, listings_summary = combine_airbnb_files(
    Listing_Data_Path,
    "listings"
)

In [ ]:
#Validate integration
print("\n========== DATA VALIDATION ==========")

# Confirm input row counts were preserved
assert len(listings_master) == listings_summary["rows"].sum()
print("All input rows preserved successfully.")

# Check unmapped locations
print("\nUnmapped listing locations:")
print(
    listings_master.loc[
        listings_master["country"] == "Unknown",
        "location"
    ].unique()
)

# Check duplicate IDs within a location
print("\nDuplicate listing IDs:")
print(
    listings_master.duplicated(
        subset=["location", "id"]
    ).sum()
)

# Check missing values
print("\nTop 15 missing values - Listings:")
display(
    listings_master.isna()
    .sum()
    .sort_values(ascending=False)
    .head(15)
)


In [ ]:
#Display listings summary table
print("\n========== LISTINGS BY LOCATION ==========")
display(
    listings_master.groupby(
        ["country", "location"]
    ).size().reset_index(name="listing_count")
    .sort_values("listing_count", ascending=False)
)


In [ ]:
#Save master listings dataset
listings_master.to_parquet(
    Path(Output_Data_Path) / "listings_master.parquet",
    index=False
)

# Save a record of source files
listings_summary.to_csv(
    Path(Output_Data_Path) / "listings_file_summary.csv",
    index=False
)

print("\nMaster listings dataset saved successfully!")

In [ ]:
#Display final listings dataframe
print("\nLISTINGS MASTER:")
print(listings_master.shape)
display(listings_master.head())

2. Reviews:

In [ ]:
# Discover reviews files (reuse the existing DuckDB connection from the setup cell).
output_folder = Path(Output_Data_Path)
output_folder.mkdir(parents=True, exist_ok=True)
reviews_files = sorted(Path(Review_Data_Path).glob('*_reviews.csv'))
print(f'Found {len(reviews_files)} review files')
if not reviews_files:
    raise FileNotFoundError(f'No *_reviews.csv files found in {Review_Data_Path}')

# CAUTION: This resets the raw review integration table when rerunning the preparation.
con.execute('DROP TABLE IF EXISTS reviews_master')

In [ ]:
#Process and integrate files
for i, file in enumerate(reviews_files, start=1):

    location = (
        file.stem
        .removesuffix("_reviews")
        .replace("_", " ")
    )

    country = location_country.get(location, "Unknown")

    print(
        f"[{i}/{len(reviews_files)}] "
        f"Processing {location}, {country}"
    )

    if i == 1:
        con.execute("""
            CREATE TABLE reviews_master AS
            SELECT
                CAST(id AS VARCHAR) AS id,
                CAST(listing_id AS VARCHAR) AS listing_id,
                TRY_CAST(date AS DATE) AS date,
                CAST(reviewer_id AS VARCHAR) AS reviewer_id,
                reviewer_name,
                comments,
                ? AS location,
                ? AS country,
                ? AS source_file
            FROM read_csv_auto(
                ?,
                all_varchar = true
            )
        """, [location, country, file.name, str(file)])

    else:
        con.execute("""
            INSERT INTO reviews_master
            SELECT
                CAST(id AS VARCHAR),
                CAST(listing_id AS VARCHAR),
                TRY_CAST(date AS DATE),
                CAST(reviewer_id AS VARCHAR),
                reviewer_name,
                comments,
                ? AS location,
                ? AS country,
                ? AS source_file
            FROM read_csv_auto(
                ?,
                all_varchar = true
            )
        """, [location, country, file.name, str(file)])

print("All review files integrated.")

In [ ]:
#Validate reviews integration
reviews_summary = con.execute("""
    SELECT
        country,
        location,
        COUNT(*) AS review_count
    FROM reviews_master
    GROUP BY country, location
    ORDER BY review_count DESC
""").df()

display(reviews_summary)

total_reviews = con.execute("""
    SELECT COUNT(*) FROM reviews_master
""").fetchone()[0]

print(f"Total reviews: {total_reviews:,}")

In [ ]:
# Export original integrated reviews to Parquet.
output_file = Path(Output_Data_Path) / 'reviews_master.parquet'
safe_path = str(output_file).replace("'", "''")
con.execute(f"COPY reviews_master TO '{safe_path}' (FORMAT PARQUET, COMPRESSION ZSTD)")
print(f'Saved to: {output_file}')

In [ ]:
#Display final reviews dataframe
reviews_aggregated = con.execute("""
    SELECT
        location,
        listing_id,

        COUNT(*) AS total_reviews,

        MIN(date) AS first_review_date,
        MAX(date) AS latest_review_date,

        COUNT(DISTINCT reviewer_id) AS unique_reviewers,

        COUNT(*) FILTER (
            WHERE date >= DATE '2025-01-01'
        ) AS reviews_since_2025

    FROM reviews_master

    GROUP BY location, listing_id
""").df()

display(reviews_aggregated.head())

In [ ]:
reviews_overview = con.execute("""
    SELECT
        COUNT(*) AS total_reviews,
        COUNT(DISTINCT id) AS unique_review_ids,
        COUNT(DISTINCT listing_id) AS unique_listing_ids,
        COUNT(DISTINCT location) AS total_locations,
        COUNT(DISTINCT country) AS total_countries
    FROM reviews_master
""").df()

print("\nReviews overview:")
display(reviews_overview)


In [ ]:
#Check reviews for duplications
duplicate_summary = con.execute("""
    SELECT
        COUNT(*) AS total_reviews,
        COUNT(DISTINCT id) AS unique_review_ids,
        COUNT(*) - COUNT(DISTINCT id) AS duplicate_id_count
    FROM reviews_master
""").df()

print("\nDuplicate review IDs:")
display(duplicate_summary)

#Check for duplicated reviews
duplicate_reviews = con.execute("""
    SELECT
        id,
        COUNT(*) AS occurrences,
        COUNT(DISTINCT location) AS locations
    FROM reviews_master
    WHERE id IS NOT NULL
    GROUP BY id
    HAVING COUNT(*) > 1
    ORDER BY occurrences DESC
    LIMIT 20
""").df()

print("\nDuplicate reviews:")
display(duplicate_reviews)

#Check for location duplicates
location_duplicates = con.execute("""
    SELECT
        location,
        id,
        COUNT(*) AS occurrences
    FROM reviews_master
    WHERE id IS NOT NULL
    GROUP BY location, id
    HAVING COUNT(*) > 1
    ORDER BY occurrences DESC
    LIMIT 20
""").df()

print("\nLocation duplicates:")
display(location_duplicates)

In [ ]:
#Check that duplicate IDs have matching content
duplicate_examples = con.execute("""
    SELECT *
    FROM reviews_master
    WHERE id IN (
        SELECT id
        FROM reviews_master
        WHERE id IS NOT NULL
        GROUP BY id
        HAVING COUNT(*) > 1
    )
    ORDER BY id, location
    LIMIT 30
""").df()

print("\nDuplicate examples:")
display(duplicate_examples)

In [ ]:
#Check reviews for missing values
missing_values = con.execute("""
    SELECT
        COUNT(*) AS total_rows,

        COUNT(*) FILTER (WHERE id IS NULL)
            AS missing_review_id,

        COUNT(*) FILTER (WHERE listing_id IS NULL)
            AS missing_listing_id,

        COUNT(*) FILTER (WHERE date IS NULL)
            AS missing_date,

        COUNT(*) FILTER (WHERE reviewer_id IS NULL)
            AS missing_reviewer_id,

        COUNT(*) FILTER (WHERE reviewer_name IS NULL)
            AS missing_reviewer_name,

        COUNT(*) FILTER (WHERE comments IS NULL)
            AS missing_comments,

        COUNT(*) FILTER (WHERE location IS NULL)
            AS missing_location,

        COUNT(*) FILTER (WHERE country IS NULL)
            AS missing_country,

        COUNT(*) FILTER (WHERE source_file IS NULL)
            AS missing_source_file

    FROM reviews_master
""").df()

print("\nMissing values:")
display(missing_values.T)

columns = [
    row[1]
    for row in con.execute(
        "PRAGMA table_info('reviews_master')"
    ).fetchall()
]

missing_queries = []

for col in columns:
    safe_col = '"' + col.replace('"', '""') + '"'

    missing_queries.append(f"""
        SELECT
            '{col.replace("'", "''")}' AS column_name,
            COUNT(*) FILTER (
                WHERE {safe_col} IS NULL
            ) AS missing_count,
            ROUND(
                100.0 * COUNT(*) FILTER (
                    WHERE {safe_col} IS NULL
                ) / NULLIF(COUNT(*), 0),
                2
            ) AS missing_percentage
        FROM reviews_master
    """)

missing_sql = "\nUNION ALL\n".join(missing_queries)

missing_summary = con.execute(f"""
    SELECT *
    FROM ({missing_sql})
    ORDER BY missing_percentage DESC
""").df()

print("\nMissing values by column:")
display(missing_summary)

#Check missing values by location
missing_by_location = con.execute("""
    SELECT
        country,
        location,
        COUNT(*) AS total_reviews,

        COUNT(*) FILTER (
            WHERE comments IS NULL
        ) AS missing_comments,

        ROUND(
            100.0 * COUNT(*) FILTER (
                WHERE comments IS NULL
            ) / NULLIF(COUNT(*), 0),
            2
        ) AS missing_comments_pct,

        COUNT(*) FILTER (
            WHERE date IS NULL
        ) AS missing_dates

    FROM reviews_master
    GROUP BY country, location
    ORDER BY missing_comments_pct DESC
""").df()

print("\nMissing comments by location:")
display(missing_by_location)

#Check for empty strings
empty_comments = con.execute("""
    SELECT
        COUNT(*) AS total_reviews,

        COUNT(*) FILTER (
            WHERE comments IS NULL
        ) AS null_comments,

        COUNT(*) FILTER (
            WHERE comments IS NOT NULL
            AND TRIM(comments) = ''
        ) AS empty_comments,

        COUNT(*) FILTER (
            WHERE comments IS NOT NULL
            AND TRIM(comments) <> ''
        ) AS non_empty_comments

    FROM reviews_master
""").df()

print("\nEmpty comments:")
display(empty_comments)

In [ ]:
# Save data-quality summaries locally instead of to Google Drive.
EDA_DIR.mkdir(parents=True, exist_ok=True)
missing_summary.to_csv(EDA_DIR / 'reviews_missing_values.csv', index=False)
duplicate_summary.to_csv(EDA_DIR / 'reviews_duplicate_summary.csv', index=False)
missing_by_location.to_csv(EDA_DIR / 'reviews_missing_by_location.csv', index=False)
print('Review data-quality summaries saved:', EDA_DIR)

In [ ]:
#Remove duplicated reviews
con.execute("""
    CREATE OR REPLACE TEMP TABLE duplicate_ids AS
    SELECT id
    FROM reviews_master
    GROUP BY id
    HAVING COUNT(*) > 1
""")

display(con.execute("""
    SELECT COUNT(*) AS duplicated_id_groups
    FROM duplicate_ids
""").df())

con.execute("""
    CREATE OR REPLACE TEMP TABLE reviews_to_keep AS
    SELECT * EXCLUDE (row_num)
    FROM (
        SELECT
            r.*,
            ROW_NUMBER() OVER (
                PARTITION BY r.id
                ORDER BY
                    r.location,
                    r.listing_id,
                    r.source_file
            ) AS row_num
        FROM reviews_master r
        INNER JOIN duplicate_ids d
            ON r.id = d.id
    )
    WHERE row_num = 1
""")

output_folder = Path(Output_Data_Path)

output_folder.mkdir(parents=True, exist_ok=True)

output_file = output_folder / "reviews_clean.parquet"

# Escape the file path for SQL
safe_path = str(output_file).replace("'", "''")

# Export cleaned reviews directly to local Parquet
con.execute(f"""
    COPY (
        SELECT r.*
        FROM reviews_master r
        WHERE NOT EXISTS (
            SELECT 1
            FROM duplicate_ids d
            WHERE d.id = r.id
        )

        UNION ALL BY NAME

        SELECT *
        FROM reviews_to_keep
    )
    TO '{safe_path}'
    (FORMAT PARQUET, COMPRESSION ZSTD)
""")

print(f"Cleaned reviews saved to: {output_file}")

In [ ]:
# Verify that cleaning retained exactly one record per non-null review ID.
cleaned_count = con.execute(
    'SELECT COUNT(*) FROM read_parquet(?)', [str(output_file)]
).fetchone()[0]
original_count = con.execute('SELECT COUNT(*) FROM reviews_master').fetchone()[0]
unique_review_ids = con.execute(
    'SELECT COUNT(DISTINCT id) FROM reviews_master'
).fetchone()[0]
null_review_ids = con.execute(
    'SELECT COUNT(*) FROM reviews_master WHERE id IS NULL'
).fetchone()[0]

print(f'Original reviews: {original_count:,}')
print(f'Cleaned reviews: {cleaned_count:,}')
print(f'Duplicates removed: {original_count - cleaned_count:,}')
# Source cleaning deduplicates non-null IDs, retaining null-ID records if any.
assert cleaned_count == unique_review_ids + null_review_ids
assert original_count >= cleaned_count
print('Review ID deduplication validation passed.')

# Make the output accessible in DuckDB for follow-on analysis.
safe_path = str(output_file).replace("'", "''")
con.execute(f"CREATE OR REPLACE VIEW reviews_clean AS SELECT * FROM read_parquet('{safe_path}')")
print('Processed artifacts:', Output_Data_Path)